# Predicción de Consumo de Alcohol — Rusia 2024

Red neuronal profunda para predecir el consumo de alcohol puro per cápita en regiones de Rusia.

▶ **Ejecutar `Entorno → Tiempo de ejecución → Ejecutar todo`** — no requiere configuración.

## 1. Instalar dependencias

In [ ]:
!pip install -q torch pandas numpy scikit-learn matplotlib seaborn  # Instala dependencias silenciosamente
print("Dependencias listas")  # Confirma instalación

## 2. Importar librerías

In [ ]:
import warnings  # Suprime warnings no críticos
warnings.filterwarnings("ignore")

import pandas as pd  # pandas: manipulación de DataFrames
import numpy as np  # numpy: operaciones numéricas y álgebra lineal
import matplotlib.pyplot as plt  # matplotlib: gráficas estáticas
import seaborn as sns  # seaborn: gráficas estadísticas (sobre matplotlib)
import torch  # PyTorch: framework de deep learning
import torch.nn as nn  # nn: capas, activaciones, dropout
from torch.utils.data import Dataset, DataLoader  # Dataset: manejo de lotes y muestras
import torch.optim as optim  # optim: optimizadores (Adam, SGD)
from sklearn.preprocessing import OneHotEncoder, StandardScaler  # OneHotEncoder: categóricas a vectores binarios
from sklearn.model_selection import train_test_split  # train_test_split: división train/val/test
from sklearn.linear_model import LinearRegression  # LinearRegression: baseline del modelo
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score  # MSE, MAE, R²: métricas de regresión

sns.set_theme(style="whitegrid")  # Tema visual: fondo blanco con cuadrícula
print("Importaciones listas")  # Confirma que todas las importaciones funcionaron


## 3. Definir clases del pipeline

Todas las clases necesarias encapsuladas aquí.

In [ ]:
class CSVLoader:
    @staticmethod
    def load(url):
        encodings = ["utf-8", "latin-1", "cp1252", "iso-8859-1"]
        for enc in encodings:
            try:
                df = pd.read_csv(url, encoding=enc)
                break
            except (UnicodeDecodeError, UnicodeError):
                continue
        else:
            df = pd.read_csv(url, encoding="latin-1", encoding_errors="replace")
        df["Type of alcoholic beverages"] = (
            df["Type of alcoholic beverages"]
            .str.replace(r"[^A-Za-z]ider", "Cider", regex=True)
        )
        return df


class DataPreprocessor:
    def __init__(self):
        self.cat_encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
        self.scaler = StandardScaler()
        self.cat_cols_ = ["Region", "Type of alcoholic beverages"]
        self.num_col_count_ = None
        self.fitted_ = False

    def pivot_data(self, df):
        df = df.rename(columns={
            "Consumption of alcoholic beverages (thousands of decaliters)": "vol",
            "Consumption of alcoholic beverages (in liters per capita)": "cap",
            "Consumption of alcoholic beverages (in liters of pure alcohol per capita)": "alc",
        })
        pivoted = df.pivot_table(
            index=["Region", "Type of alcoholic beverages"],
            columns="Year",
            values=["vol", "cap", "alc"],
        )
        pivoted.columns = [f"{metric}_{year}" for metric, year in pivoted.columns]
        pivoted = pivoted.reset_index()
        return pivoted

    def extract_train_data(self, pivoted):
        feature_cols = [f"{m}_{y}" for m in ["vol", "cap", "alc"] for y in range(2017, 2023)]
        X = pivoted[self.cat_cols_ + feature_cols]
        y = pivoted["alc_2023"]
        return X, y

    def extract_predict_data(self, pivoted):
        feature_cols = [f"{m}_{y}" for m in ["vol", "cap", "alc"] for y in range(2018, 2024)]
        X = pivoted[self.cat_cols_ + feature_cols]
        return X

    def _get_num_cols(self, X):
        return [c for c in X.columns if c not in self.cat_cols_]

    def fit(self, X):
        num_cols = self._get_num_cols(X)
        self.cat_encoder.fit(X[self.cat_cols_].values)
        self.scaler.fit(X[num_cols].values)
        self.num_col_count_ = len(num_cols)
        self.fitted_ = True

    def transform(self, X):
        cat_encoded = self.cat_encoder.transform(X[self.cat_cols_].values)
        num_cols = self._get_num_cols(X)
        num_scaled = self.scaler.transform(X[num_cols].values)
        return np.hstack([cat_encoded, num_scaled])

    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)


class DataSplitter:
    def __init__(self, test_size=0.15, val_size=0.15, random_state=42):
        self.test_size = test_size
        self.val_size = val_size
        self.random_state = random_state

    def split(self, X, y):
        X_temp, X_test, y_temp, y_test = train_test_split(
            X, y, test_size=self.test_size, random_state=self.random_state
        )
        val_frac = self.val_size / (1 - self.test_size)
        X_train, X_val, y_train, y_val = train_test_split(
            X_temp, y_temp, test_size=val_frac, random_state=self.random_state
        )
        return {
            "X_train": X_train, "X_val": X_val, "X_test": X_test,
            "y_train": y_train, "y_val": y_val, "y_test": y_test,
            "train_idx": X_train.index,
            "val_idx": X_val.index,
            "test_idx": X_test.index,
        }


class AlcoholDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).view(-1, 1) if y is not None else None

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        if self.y is not None:
            return self.X[idx], self.y[idx]
        return self.X[idx]


class AlcoholPredictor(nn.Module):
    def __init__(self, input_dim, hidden_dims=(128, 64), dropout=0.3):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev_dim = hidden_dim
        layers.append(nn.Linear(prev_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class EarlyStopping:
    def __init__(self, patience=15, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = float("inf")
        self.best_state = None

    def __call__(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.best_state = {k: v.clone() for k, v in model.state_dict().items()}
            self.counter = 0
            return False
        self.counter += 1
        return self.counter >= self.patience

    def restore(self, model):
        if self.best_state is not None:
            model.load_state_dict(self.best_state)


class Trainer:
    def __init__(self, model, train_dataset, val_dataset,
                 lr=0.001, weight_decay=1e-5, batch_size=32, max_epochs=200, patience=15):
        self.model = model
        self.train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        self.val_loader = DataLoader(val_dataset, batch_size=batch_size)
        self.optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
        self.criterion = nn.MSELoss()
        self.early_stopping = EarlyStopping(patience=patience)
        self.max_epochs = max_epochs

    def train(self):
        history = {"train_loss": [], "val_loss": []}
        for epoch in range(1, self.max_epochs + 1):
            train_loss = self._run_epoch(training=True)
            val_loss = self._run_epoch(training=False)
            history["train_loss"].append(train_loss)
            history["val_loss"].append(val_loss)
            if epoch % 10 == 0 or epoch == 1:
                print(f"Epoch {epoch:3d} | Train MSE: {train_loss:.6f} | Val MSE: {val_loss:.6f}")
            if val_loss == min(history["val_loss"]):
                self.best_state = {k: v.clone() for k, v in self.model.state_dict().items()}
            if self.early_stopping(val_loss, self.model):
                print(f"Early stopping at epoch {epoch}")
                break
        self.early_stopping.restore(self.model)
        return history

    def _run_epoch(self, training):
        self.model.train(training)
        loader = self.train_loader if training else self.val_loader
        total_loss = 0.0
        with torch.set_grad_enabled(training):
            for X_batch, y_batch in loader:
                if training:
                    self.optimizer.zero_grad()
                pred = self.model(X_batch)
                loss = self.criterion(pred, y_batch)
                if training:
                    loss.backward()
                    self.optimizer.step()
                total_loss += loss.item()
        return total_loss / len(loader)


class Evaluator:
    def __init__(self):
        self.metrics = {
            "MSE": mean_squared_error,
            "MAE": mean_absolute_error,
            "R²": r2_score,
        }

    def evaluate(self, model, dataset):
        loader = DataLoader(dataset, batch_size=64)
        model.eval()
        all_preds, all_true = [], []
        with torch.no_grad():
            for X_batch, y_batch in loader:
                all_preds.append(model(X_batch).numpy())
                all_true.append(y_batch.numpy())
        y_pred = np.vstack(all_preds).flatten()
        y_true = np.vstack(all_true).flatten()
        return {name: metric(y_true, y_pred) for name, metric in self.metrics.items()}

    def evaluate_sklearn(self, model, X_test, y_test):
        y_pred = model.predict(X_test)
        return {name: metric(y_test, y_pred) for name, metric in self.metrics.items()}


class Predictor:
    def __init__(self, preprocessor, model):
        self.preprocessor = preprocessor
        self.model = model

    def predict_2024(self, pivoted):
        X_2024 = self.preprocessor.extract_predict_data(pivoted)
        X_processed = self.preprocessor.transform(X_2024)
        dataset = AlcoholDataset(X_processed)
        loader = DataLoader(dataset, batch_size=64)
        self.model.eval()
        predictions = []
        with torch.no_grad():
            for X_batch in loader:
                predictions.append(self.model(X_batch).numpy())
        y_pred = np.vstack(predictions).flatten()
        y_pred = np.maximum(y_pred, 0)
        result = pivoted[["Region", "Type of alcoholic beverages"]].copy()
        result["Predicted_2024"] = y_pred
        return result

    def ranking_by_region(self, predictions):
        return (
            predictions.groupby("Region")["Predicted_2024"]
            .mean().sort_values(ascending=False)
            .reset_index()
            .rename(columns={"Predicted_2024": "Avg_Pure_Alcohol_2024"})
        )

    def ranking_by_beverage(self, predictions):
        return (
            predictions.groupby("Type of alcoholic beverages")["Predicted_2024"]
            .mean().sort_values(ascending=False)
            .reset_index()
            .rename(columns={"Predicted_2024": "Avg_Pure_Alcohol_2024"})
        )


print("Clases definidas")

## 4. Cargar datos desde GitHub

In [ ]:
URL = "https://raw.githubusercontent.com/jfelipeq14/alcohol-prediction-russia/main/Consumption%20of%20alcoholic%20beverages%202017-2023%20(Pivot%20table).csv"  # URL directa al dataset en GitHub (raw)
df = CSVLoader.load(URL)  # Carga CSV con detección automática de encoding
print(f"Filas: {len(df):,} | Columnas: {len(df.columns)}")  # Muestra dimensiones del dataset
print(f"Tipos de bebida: {list(df['Type of alcoholic beverages'].unique())}")  # Lista los 7 tipos de bebida disponibles
print(f"Regiones: {df['Region'].nunique()}")  # Total de regiones (85)
print(f"Años: {sorted(df['Year'].unique())}")  # Años disponibles en el dataset (2017-2023)

## 5. Preprocesar (pivot, split, encode, scale)

In [ ]:
preprocessor = DataPreprocessor()  # Instancia el preprocesador
pivoted = preprocessor.pivot_data(df)  # Pivota de formato largo a ancho (18 cols numéricas)
print(f"Muestras (región × bebida): {len(pivoted)}")  # 85 regiones x 7 bebidas = 595 muestras

X, y = preprocessor.extract_train_data(pivoted)  # Separa features (18 numéricas + 2 categóricas) del target
print(f"Features: {X.shape[1]} | Target: {y.name}")  # Target: consumo 2023

splitter = DataSplitter()  # Splitter con proporciones fijas 70/15/15
splits = splitter.split(X, y)  # Ejecuta la división estratificada
print(f"Train: {len(splits['X_train'])} | Val: {len(splits['X_val'])} | Test: {len(splits['X_test'])}")  # Train=415 | Val=90 | Test=90

X_train = preprocessor.fit_transform(splits["X_train"])  # Ajusta encoder+scaler en train y transforma
X_val = preprocessor.transform(splits["X_val"])  # Transforma val con parámetros de train (sin reajustar)
X_test = preprocessor.transform(splits["X_test"])  # Transforma test con parámetros de train
y_train = splits["y_train"].values  # Convierte a arreglo numpy
y_val = splits["y_val"].values  # ...
y_test = splits["y_test"].values  # ...

train_dataset = AlcoholDataset(X_train, y_train)  # Envuelve tensores en Dataset de PyTorch
val_dataset = AlcoholDataset(X_val, y_val)  # Dataset de validación
test_dataset = AlcoholDataset(X_test, y_test)  # Dataset de test

input_dim = X_train.shape[1]  # Dimensión real de entrada al modelo
print(f"Dimensión de entrada: {input_dim}")  # Muestra la dimensión final (~110)

## 6. Construir y entrenar red neuronal

In [ ]:
model = AlcoholPredictor(input_dim=input_dim, hidden_dims=[128, 64], dropout=0.3)  # Red neuronal: 110->128->64->1 con Dropout 0.3
total_params = sum(p.numel() for p in model.parameters())  # Suma todos los parámetros entrenables
print(f"Arquitectura: {input_dim} → [128, 64] → 1")  # Muestra la arquitectura completa
print(f"Parámetros: {total_params:,}")  # Total de parámetros: 22,529

trainer = Trainer(model, train_dataset, val_dataset,  # Trainer con Adam (lr=0.001), weight decay 1e-5, batch=32
                  lr=0.001, weight_decay=1e-5, batch_size=32,  # Early stopping: patience 15 épocas
                  max_epochs=200, patience=15)  # Ejecuta el ciclo de entrenamiento completo
history = trainer.train()  # Mejor pérdida en validación durante el entrenamiento
print(f"\nMejor validation MSE: {min(history['val_loss']):.6f}")

## 7. Evaluar (NN vs Linear Regression)

In [ ]:
evaluator = Evaluator()  # Instancia el evaluador de métricas
nn_metrics = evaluator.evaluate(model, test_dataset)  # Evalúa la red neuronal en el conjunto de test
print("── Neural Network ──")  # Encabezado de resultados de la red neuronal
for name, val in nn_metrics.items():  # Itera sobre MSE, MAE, R²
    print(f"  {name}: {val:.4f}")

baseline = LinearRegression()  # Baseline: modelo de regresión lineal
baseline.fit(X_train, y_train)  # Evalúa la regresión lineal en test
base_metrics = evaluator.evaluate_sklearn(baseline, X_test, y_test)  # Encabezado de resultados de la línea base
print("\n── Linear Regression ──")
for name, val in base_metrics.items():
    print(f"  {name}: {val:.4f}")

## 8. Predecir 2024

In [ ]:
predictor = Predictor(preprocessor, model)  # Predictor con ventana deslizante 2018->2023
predictions = predictor.predict_2024(pivoted)  # Predice consumo de alcohol puro para 2024

region_ranking = predictor.ranking_by_region(predictions)  # Ranking: promedio por región
beverage_ranking = predictor.ranking_by_beverage(predictions)  # Ranking: promedio por tipo de bebida

print(f"Predicciones: {len(predictions)} filas")  # Muestra las primeras 10 predicciones
display(predictions.head(10))
  # Top 10 regiones con mayor consumo estimado
print("\n── Ranking por región (top 10) ──")
display(region_ranking.head(10))
  # Ranking completo de consumo por bebida
print("\n── Ranking por bebida ──")
display(beverage_ranking)

## 9. Resultados

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))  # Crea figura de 10x5 pulgadas
ax.plot(history["train_loss"], label="Train MSE", alpha=0.8)  # Curva de pérdida (MSE) en entrenamiento
ax.plot(history["val_loss"], label="Val MSE", alpha=0.8)  # Curva de pérdida (MSE) en validación
best_epoch = np.argmin(history["val_loss"]) + 1  # Encuentra la época con menor pérdida en validación
ax.axvline(best_epoch - 1, color="gray", linestyle="--", alpha=0.5, label=f"Mejor época ({best_epoch})")  # Línea vertical punteada en la mejor época
ax.set_xlabel("Época")  # Etiqueta del eje X
ax.set_ylabel("MSE")  # Etiqueta del eje Y
ax.set_title("Historial de entrenamiento")  # Título del gráfico
ax.legend()  # Muestra la leyenda con ambas curvas
plt.tight_layout()  # Ajusta márgenes para evitar solapamientos
plt.show()  # Renderiza la figura en el notebook

In [ ]:
y_test_pred_nn = model(torch.tensor(X_test, dtype=torch.float32)).detach().numpy().flatten()  # Predice con la red neuronal sobre X_test
y_test_pred_base = baseline.predict(X_test)  # Predice con la regresión lineal sobre X_test

fig, axes = plt.subplots(1, 2, figsize=(14, 5))  # Dos paneles lado a lado (14x5 pulgadas)
for ax, y_pred, title in zip(  # Itera sobre ambos modelos: NN y LR
    axes,
    [y_test_pred_nn, y_test_pred_base],
    ["Neural Network", "Linear Regression"],  # Dispersión: valor real (x) vs predicho (y)
):  # Límites del gráfico: mínimo y máximo de ambos ejes
    ax.scatter(y_test, y_pred, alpha=0.6, edgecolors="k", linewidth=0.5)  # Línea diagonal roja: predicción perfecta (y=x)
    lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]  # Etiqueta del eje X
    ax.plot(lims, lims, "r--", alpha=0.7)  # Etiqueta del eje Y
    ax.set_xlabel("Real")  # Título del panel: nombre del modelo
    ax.set_ylabel("Predicho")
    ax.set_title(title)  # Ajusta márgenes
plt.tight_layout()  # Renderiza la figura
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))  # Crea figura de 10x4 pulgadas
x = range(len(nn_metrics))  # Índices para las 3 métricas: 0, 1, 2
nn_vals = list(nn_metrics.values())  # Valores de la red neuronal [MSE, MAE, R²]
base_vals = list(base_metrics.values())  # Valores de la regresión lineal [MSE, MAE, R²]
ax.bar(x, nn_vals, width=0.35, label="Neural Network", color="steelblue", alpha=0.8)  # Barras azules para NN (izquierda, ancho 0.35)
ax.bar([i + 0.35 for i in x], base_vals, width=0.35, label="Linear Regression", color="coral", alpha=0.8)  # Barras naranjas para LR (derecha, desplazadas +0.35)
ax.set_xticks([i + 0.175 for i in x])  # Coloca etiquetas centradas entre ambas barras
ax.set_xticklabels(list(nn_metrics.keys()))  # Nombres de las métricas en el eje X
ax.set_title("Comparación de métricas")  # Título del gráfico
ax.legend()  # Muestra leyenda: azul=NN, naranja=LR
for bars in [nn_vals, base_vals]:  # Itera sobre las barras de ambos modelos
    for i, v in enumerate(bars):  # Anota el valor numérico sobre cada barra
        offset = 0.175 if bars is nn_vals else 0.525  # Ajusta posición horizontal del texto
        ax.text(i + offset - 0.175, v + 0.01, f"{v:.3f}", ha="center", fontsize=9)  # Desplaza según sea barra izquierda o derecha
plt.tight_layout()
plt.show()  # Ajusta márgenes

### 9.4 Análisis de residuales

In [ ]:
residuals = y_test - y_test_pred_nn  # Error = valor real - valor predicho por NN
pivoted_test = pivoted.loc[splits["test_idx"]]  # Filas de test con sus metadatos originales
beverage_types_test = pivoted_test["Type of alcoholic beverages"].values  # Tipo de bebida de cada muestra de test

fig, axes = plt.subplots(1, 3, figsize=(16, 5))  # Tres paneles horizontales (16x5 pulgadas)

# 1. Distribution + normal curve  # Histograma de residuales (densidad, no frecuencia)
axes[0].hist(residuals, bins=20, density=True, alpha=0.6, color="steelblue", edgecolor="white")  # Media y desviación estándar de los residuales
mu, std = np.mean(residuals), np.std(residuals)  # 100 puntos para dibujar la curva normal
x = np.linspace(residuals.min(), residuals.max(), 100)  # Función de densidad normal con mu y sigma empíricos
pdf = (1 / (std * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - mu) / std) ** 2)  # Curva normal ajustada (roja punteada)
axes[0].plot(x, pdf, "r--", alpha=0.7, label=f"N({mu:.3f}, {std:.3f})")  # Etiqueta del eje X
axes[0].set_xlabel("Residual")  # Etiqueta del eje Y
axes[0].set_ylabel("Densidad")  # Título del panel
axes[0].set_title("Distribución de residuales")  # Muestra la leyenda: N(mu, sigma)
axes[0].legend()
  # Dispersión: valor predicho (x) vs error (y)
# 2. Residuals vs Predicted  # Línea horizontal roja en error = 0
axes[1].scatter(y_test_pred_nn, residuals, alpha=0.6, edgecolors="k", linewidth=0.5)  # Etiqueta del eje X
axes[1].axhline(0, color="red", linestyle="--", alpha=0.5)  # Etiqueta del eje Y
axes[1].set_xlabel("Valor predicho")  # Título del panel
axes[1].set_ylabel("Residual")
axes[1].set_title("Residuales vs Valores Predichos")  # Boxplot: distribución del error por tipo de bebida
  # Línea de referencia en error = 0
# 3. Boxplot by beverage  # Título del panel
sns.boxplot(x=beverage_types_test, y=residuals, ax=axes[2], palette="Set2")  # Rota las etiquetas del eje X 15° para legibilidad
axes[2].axhline(0, color="red", linestyle="--", alpha=0.4)
axes[2].set_title("Residuales por tipo de bebida")  # Ajusta márgenes
axes[2].tick_params(axis="x", rotation=15)  # Renderiza

plt.tight_layout()
plt.show()

In [ ]:
year_cols = [f"alc_{y}" for y in range(2017, 2024)]  # Columnas de consumo puro: alc_2017 a alc_2023
corr_data = pivoted[year_cols].dropna()  # Selecciona solo esas 7 columnas y elimina filas nulas
corr_matrix = corr_data.corr()  # Matriz de correlación de Pearson (7x7)

fig, ax = plt.subplots(figsize=(8, 6))  # Crea figura de 8x6 pulgadas
sns.heatmap(corr_matrix, annot=True, fmt=".3f", cmap="coolwarm",  # Mapa de calor con valores, 3 decimales, escala rojo/azul
            vmin=-1, vmax=1, center=0, ax=ax)  # Rango fijo -1 a 1, blanco en 0 (sin correlación)
ax.set_title("Correlación entre años")  # Título del heatmap
plt.tight_layout()  # Ajusta márgenes
plt.show()  # Renderiza

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))  # Crea figura de 8x4 pulgadas
colors = plt.cm.Set2(np.linspace(0, 1, len(beverage_ranking)))  # 7 colores pastel de la paleta Set2 (una por bebida)
ax.barh(beverage_ranking["Type of alcoholic beverages"],  # Barras horizontales: tipo de bebida (y) vs litros (x)
        beverage_ranking["Avg_Pure_Alcohol_2024"], color=colors)
ax.set_xlabel("Litros de alcohol puro per cápita")  # Etiqueta del eje X
ax.set_title("Consumo estimado 2024 por tipo de bebida")  # Título del gráfico
for i, v in enumerate(beverage_ranking["Avg_Pure_Alcohol_2024"]):  # Anota el valor numérico al final de cada barra
    ax.text(v + 0.01, i, f"{v:.3f}", va="center", fontsize=10)
plt.tight_layout()  # Ajusta márgenes
plt.show()  # Renderiza

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))  # Dos paneles horizontales (16x7 pulgadas)
  # Número de regiones a mostrar en cada panel
top_n = 15  # Top 15: las 15 regiones con mayor consumo
region_ranking.head(top_n).set_index("Region").plot(
    kind="bar", ax=axes[0], color="steelblue", legend=False
)  # Título del panel izquierdo
axes[0].set_title(f"Top {top_n} regiones — Consumo estimado 2024")  # Etiqueta del eje Y (común a ambos paneles)
axes[0].set_ylabel("Litros per cápita")  # Rota etiquetas del eje X 45°
axes[0].tick_params(axis="x", rotation=45)
  # Bottom 15: las 15 regiones con menor consumo
region_ranking.tail(top_n).set_index("Region").plot(
    kind="bar", ax=axes[1], color="coral", legend=False
)  # Título del panel derecho
axes[1].set_title(f"Bottom {top_n} regiones — Consumo estimado 2024")  # Etiqueta del eje Y
axes[1].set_ylabel("Litros per cápita")  # Rota etiquetas 45°
axes[1].tick_params(axis="x", rotation=45)
  # Ajusta márgenes
plt.tight_layout()  # Renderiza
plt.show()

## 10. Comparación 2023 vs 2024

Merge de `alc_2023` (real) con `Predicted_2024` para evaluar cambios estimados.

In [ ]:
comp = pivoted[["Region", "Type of alcoholic beverages", "alc_2023"]].copy()  # Copia Región, Bebida y consumo real de 2023
comp = comp.merge(predictions, on=["Region", "Type of alcoholic beverages"])  # Combina con las predicciones 2024 (merge por Región+Bebida)
comp["Diferencia"] = comp["Predicted_2024"] - comp["alc_2023"]  # Diferencia absoluta: Predicho_2024 - Real_2023
comp["Dif_%"] = (comp["Diferencia"] / comp["alc_2023"].replace(0, np.nan)) * 100  # Diferencia porcentual (evita división entre 0)

region_change = comp.groupby("Region").agg(  # Agrupa por región: promedia real, predicho y diferencia
    Real_2023=("alc_2023", "mean"),  # Promedio real 2023 por región
    Pred_2024=("Predicted_2024", "mean"),  # Promedio estimado 2024 por región
    Diferencia=("Diferencia", "mean"),  # Promedio de la diferencia por región
).sort_values("Diferencia", ascending=False)  # Ordena de mayor aumento a mayor disminución

up = (region_change["Diferencia"] > 0).sum()  # Cuenta regiones donde el consumo aumenta (73)
down = (region_change["Diferencia"] <= 0).sum()  # Cuenta regiones donde el consumo disminuye (12)
print(f"Regiones con aumento estimado: {up} | Disminución estimada: {down}\n")  # Muestra resumen del cambio estimado

print("Top 10 regiones — mayor aumento estimado:")  # Top 10 regiones con mayor aumento
display(region_change.head(10).style.format("{:.4f}"))

print("Top 10 regiones — mayor disminución estimada:")  # Top 10 regiones con mayor disminución
display(region_change.tail(10).sort_values("Diferencia").style.format("{:.4f}"))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))  # Crea figura cuadrada de 8x8 pulgadas
ax.scatter(comp["alc_2023"], comp["Predicted_2024"],  # Dispersión: real 2023 (x) vs predicho 2024 (y)
           alpha=0.4, edgecolors="k", linewidth=0.5)  # Transparencia 0.4 para apreciar densidad de puntos
lims = [0, comp[["alc_2023", "Predicted_2024"]].max().max() + 0.5]  # Límite del eje = valor máximo + 0.5 de margen
ax.plot(lims, lims, "r--", alpha=0.5, label="Sin cambio")  # Línea diagonal y=x: puntos sobre ella = sin cambio
ax.set_xlabel("Real 2023")  # Etiqueta de la línea de referencia
ax.set_ylabel("Predicho 2024")  # Etiqueta del eje X
ax.set_title("Comparación 2023 vs 2024")  # Etiqueta del eje Y
ax.legend()  # Título del gráfico
plt.tight_layout()  # Muestra la leyenda (línea roja = sin cambio)
plt.show()  # Ajusta márgenes

In [ ]:
print("=== Fin ===")  # Marca el final de la ejecución del pipeline
print("\n💡 Los resultados solo están disponibles durante esta sesión de Colab.")  # Los resultados son temporales (sesión de Colab)
print("   Para descargarlos: panel Archivos (📁) → download.")  # Instrucciones para descargar CSVs y PNGs